# EXP002 — Slot 2 (drop nino_34)

Struktur notebook ini sama persis dengan exp001, dengan tambahan:
- `nino_34` di-drop dari feature set (ablasi Stage 9: OOF turun 0.0344 tanpa nino_34)
- Perbaikan cek kontribusi error per pos (fix skala RMSE)
- Retrain full + build submission untuk Slot 2

**Prasyarat:** `train_fe.csv` dan `test_fe.csv` sudah ada di direktori kerja (hasil FE pipeline exp001).

## 0. Setup & Imports

In [10]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

## 1. Load Data (hasil FE pipeline)

In [11]:
train_fe = pd.read_csv("train_fe.csv", parse_dates=["datetime"])
test_fe  = pd.read_csv("test_fe.csv",  parse_dates=["datetime"])

# Pastikan kategorical columns bertipe string
train_fe["nama_pos"]       = train_fe["nama_pos"].astype(str)
train_fe["horizon_bucket"] = train_fe["horizon_bucket"].astype(str)
test_fe["nama_pos"]        = test_fe["nama_pos"].astype(str)
test_fe["horizon_bucket"]  = test_fe["horizon_bucket"].astype(str)

# PENTING: sort + reset index SEKALI di sini, supaya urutan train_fe
# konsisten dengan urutan yang dipakai run_cv_v2_with_month_breakdown()
# (fungsi itu sort+reset lagi secara lokal — kalau urutan global beda,
# oof_mask/oof_preds hasil fungsi jadi MISALIGN dengan train_fe global,
# dan itu penyebab bug RMSE ~67 kemarin).
train_fe = train_fe.sort_values("datetime").reset_index(drop=True)

print(f"Loaded: train_fe {train_fe.shape}, test_fe {test_fe.shape}")

Loaded: train_fe (84396, 34), test_fe (21780, 34)


## 2. Feature Set — SLOT 2 (nino_34 DROPPED)

In [12]:
FEATURES_ALL = [
    "bulan", "day_of_year", "hour",
    "days_since_last_valid_tma",
    "nama_pos", "latitude", "longitude",
    "tma_mean_pos", "tma_std_pos", "ac_lag1_pos",
    "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
    "rainfall_mm", "rainfall_max_24h_mm",
    "humidity_pct", "dew_point_c", "cloud_cover_pct",
    "temperature_c", "wind_speed_kmh", "wind_direction_deg",
    "pressure_msl_hpa",
    "rmm1", "rmm2", "mjo_amplitude", "mjo_phase",
    "nino_34",
    "rolling_rain_48h", "rolling_rain_72h", "rolling_rain_7d",
    "horizon_days", "horizon_bucket",
]

# ── SLOT 2: drop nino_34 (ablasi Stage 9 -> OOF turun 0.0344 tanpa nino_34) ──
FEATURES = [f for f in FEATURES_ALL if f != "nino_34"]

TARGET = "tma_mdpl"
CAT_FEATURES = ["nama_pos", "horizon_bucket"]

assert "nino_34" not in FEATURES, "nino_34 harus di-drop di Slot 2!"
print(f"Jumlah fitur Slot 2 (tanpa nino_34): {len(FEATURES)}")
print(FEATURES)

Jumlah fitur Slot 2 (tanpa nino_34): 31
['bulan', 'day_of_year', 'hour', 'days_since_last_valid_tma', 'nama_pos', 'latitude', 'longitude', 'tma_mean_pos', 'tma_std_pos', 'ac_lag1_pos', 'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'rainfall_mm', 'rainfall_max_24h_mm', 'humidity_pct', 'dew_point_c', 'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'wind_direction_deg', 'pressure_msl_hpa', 'rmm1', 'rmm2', 'mjo_amplitude', 'mjo_phase', 'rolling_rain_48h', 'rolling_rain_72h', 'rolling_rain_7d', 'horizon_days', 'horizon_bucket']


## 3. Fold Definitions (v2, locked) & Gap Protocol

In [13]:
FOLDS_V2 = [
    ("2024-04-30 18:00:00", "2024-05-01 06:00:00", "2024-07-31 18:00:00"),   # 91d
    ("2024-07-31 18:00:00", "2024-08-01 06:00:00", "2024-10-31 18:00:00"),   # 91d
    ("2024-10-31 18:00:00", "2024-11-01 06:00:00", "2025-02-28 18:00:00"),   # 119d, overlap gap
    ("2025-01-20 18:00:00", "2025-01-21 06:00:00", "2025-09-18 18:00:00"),   # 240d, full horizon
]

GAP_START = pd.Timestamp("2025-02-03 18:00:00")
GAP_END   = pd.Timestamp("2025-03-01 06:00:00")

ROLLING_COLS = {
    "rolling_rain_48h": 48,
    "rolling_rain_72h": 72,
    "rolling_rain_7d":  168,
}

HORIZON_BUCKETS = [(0, 30), (31, 90), (91, 241)]

OUTLIER_POS = ["Gunungsari", "Wonogiri Dam"]

LGB_PARAMS = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.05,
    "num_leaves":       127,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     1,
    "lambda_l1":        0.1,
    "lambda_l2":        0.1,
    "verbose":          -1,
    "seed":             42,
    "deterministic":    True,
}

print("Fold & params locked — SAMA PERSIS dengan exp001 (kontrol eksperimen).")
print("Satu-satunya variabel yang berubah: FEATURES (nino_34 di-drop).")

Fold & params locked — SAMA PERSIS dengan exp001 (kontrol eksperimen).
Satu-satunya variabel yang berubah: FEATURES (nino_34 di-drop).


## 4. CV Pipeline Functions (reused dari exp001, tidak diubah)

In [14]:
def evaluate_fold_by_horizon_bucket(y_true, y_pred, val_dates, val_start, buckets=HORIZON_BUCKETS):
    horizon_days = (val_dates - val_start).dt.days
    results = []
    for lo, hi in buckets:
        mask = (horizon_days >= lo) & (horizon_days <= hi)
        if mask.sum() == 0:
            continue
        rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
        results.append({"horizon_bucket": f"{lo}-{hi}d", "n_rows": int(mask.sum()), "rmse": round(rmse, 4)})
    return pd.DataFrame(results)


def evaluate_fold_by_month(y_true, y_pred, val_dates):
    months = val_dates.dt.month
    results = []
    for m in sorted(months.unique()):
        mask = (months == m).values
        if mask.sum() < 10:
            continue
        rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
        results.append({"bulan": m, "n_rows": int(mask.sum()), "rmse": round(rmse, 4)})
    return pd.DataFrame(results)


def run_cv_v2_with_month_breakdown(train_fe, features, cat_features, target, lgb_params,
                                     folds, gap_start, gap_end, rolling_cols):
    """CV walk-forward 4-fold, leakage-safe pos-stats per fold, gap masking,
    breakdown per bulan. EXACT COPY dari exp001 — jangan diubah supaya
    perbandingan dengan/tanpa nino_34 valid (kontrol eksperimen)."""
    train_fe = train_fe.sort_values("datetime").reset_index(drop=True)
    oof_preds = np.zeros(len(train_fe))
    oof_mask  = np.zeros(len(train_fe), dtype=bool)
    best_iterations = []
    all_month_breakdowns = []

    for fold_idx, (train_cut, val_start, val_end) in enumerate(folds):
        train_cut = pd.Timestamp(train_cut)
        val_start = pd.Timestamp(val_start)
        val_end   = pd.Timestamp(val_end)

        tr_mask  = train_fe["datetime"] <= train_cut
        val_mask = (train_fe["datetime"] >= val_start) & (train_fe["datetime"] <= val_end)

        X_tr  = train_fe.loc[tr_mask,  features].copy()
        y_tr  = train_fe.loc[tr_mask,  target]
        X_val = train_fe.loc[val_mask, features].copy()
        y_val = train_fe.loc[val_mask, target]

        # Per-fold pos stats (leakage-safe: dihitung dari train fold only)
        fold_train_df = train_fe.loc[tr_mask].copy()
        pos_stats_fold = fold_train_df.groupby("nama_pos")[target].agg(
            tma_mean_pos="mean", tma_std_pos="std"
        )
        ac_fold = fold_train_df.groupby("nama_pos").apply(
            lambda g: g.sort_values("datetime")[target].dropna().autocorr(lag=1)
        ).rename("ac_lag1_pos")
        pos_stats_fold = pos_stats_fold.join(ac_fold)

        for col in ["tma_mean_pos", "tma_std_pos", "ac_lag1_pos"]:
            if col in features:
                X_tr[col]  = train_fe.loc[tr_mask,  "nama_pos"].map(pos_stats_fold[col]).values
                X_val[col] = train_fe.loc[val_mask, "nama_pos"].map(pos_stats_fold[col]).values

        # Gap masking untuk rolling features
        for col, w in rolling_cols.items():
            if col not in features:
                continue
            corrupt = (
                (train_fe.loc[val_mask, "datetime"] >= gap_start) &
                (train_fe.loc[val_mask, "datetime"] <= gap_end + pd.Timedelta(hours=w))
            )
            X_val.loc[corrupt[corrupt].index, col] = np.nan

        for col in cat_features:
    # PENTING: categories harus SAMA antara train & val (dan nanti test),
    # kalau tidak, LightGBM salah mapping integer code -> kategori,
    # menyebabkan prediksi "nyasar" ke pos lain (bug ditemukan 12 Juli 2026).
            all_categories = pd.Categorical(train_fe[col]).categories
            X_tr[col]  = pd.Categorical(X_tr[col],  categories=all_categories)
            X_val[col] = pd.Categorical(X_val[col], categories=all_categories)

        ds_tr  = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=cat_features, free_raw_data=False)
        ds_val = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_features, free_raw_data=False)

        model = lgb.train(
            lgb_params, ds_tr, num_boost_round=2000,
            valid_sets=[ds_val],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False),
                       lgb.log_evaluation(period=200)],
        )

        best_iterations.append(model.best_iteration)
        preds = model.predict(X_val)
        oof_preds[val_mask] = preds
        oof_mask[val_mask]  = True

        fold_rmse = np.sqrt(mean_squared_error(y_val, preds))
        print(f"Fold {fold_idx+1} RMSE: {fold_rmse:.4f} | best_iter: {model.best_iteration}")

        val_dates = train_fe.loc[val_mask, "datetime"]
        month_bd = evaluate_fold_by_month(y_val.values, preds, val_dates)
        month_bd["fold"] = fold_idx + 1
        all_month_breakdowns.append(month_bd)

    oof_true = train_fe.loc[oof_mask, target].values
    oof_pred = oof_preds[oof_mask]
    oof_rmse = np.sqrt(mean_squared_error(oof_true, oof_pred))
    print(f"\n=== OOF RMSE: {oof_rmse:.4f} ===")

    all_months_df = pd.concat(all_month_breakdowns, ignore_index=True)

    return {
        "oof_rmse": oof_rmse,
        "oof_preds": oof_preds,
        "oof_mask": oof_mask,
        "month_breakdown": all_months_df,
        "best_iterations": best_iterations,
    }

print("CV functions ready (exact copy dari exp001).")

CV functions ready (exact copy dari exp001).


## 5. Jalankan CV — Slot 2 (tanpa nino_34)

Kontrol eksperimen: `LGB_PARAMS`, `FOLDS_V2`, dan fungsi CV **sama persis** dengan exp001.
Satu-satunya perbedaan adalah `FEATURES` (nino_34 di-drop). Expected OOF ~1.2709 berdasarkan ablasi sebelumnya.

In [15]:
results_slot2 = run_cv_v2_with_month_breakdown(
    train_fe=train_fe,
    features=FEATURES,
    cat_features=CAT_FEATURES,
    target=TARGET,
    lgb_params=LGB_PARAMS,
    folds=FOLDS_V2,
    gap_start=GAP_START,
    gap_end=GAP_END,
    rolling_cols=ROLLING_COLS,
)

print(f"\nOOF RMSE Slot 2 (tanpa nino_34): {results_slot2['oof_rmse']:.4f}")
print(f"Referensi — Slot 1 (dengan nino_34, exp001): 1.3052")
print(f"Referensi — ablasi sebelumnya (tanpa nino_34): 1.2709")

Fold 1 RMSE: 1.1605 | best_iter: 99
Fold 2 RMSE: 1.5035 | best_iter: 91
Fold 3 RMSE: 1.2839 | best_iter: 143
Fold 4 RMSE: 1.2965 | best_iter: 112

=== OOF RMSE: 1.3084 ===

OOF RMSE Slot 2 (tanpa nino_34): 1.3084
Referensi — Slot 1 (dengan nino_34, exp001): 1.3052
Referensi — ablasi sebelumnya (tanpa nino_34): 1.2709


## 6. Cek Kontribusi Error per Pos (FIXED — skala RMSE dikoreksi)

Perbaikan dari cell diagnostik sebelumnya: pastikan RMSE dihitung dengan `np.sqrt(mean_squared_error(...))`
secara konsisten dan dicetak per-pos dengan skala yang sama seperti OOF RMSE agregat (~1.x), bukan skala aneh (~67).
Kalau skala aneh muncul lagi, kemungkinan `oof_true`/`oof_pred` tercampur dengan kolom lain — dicek eksplisit di bawah.

In [16]:
# Cek index alignment: train_fe global vs urutan yang dipakai fungsi CV harus SAMA
assert train_fe["datetime"].is_monotonic_increasing, \
    "train_fe TIDAK ter-sort by datetime! Jalankan ulang Fix 1 di Section 1 sebelum lanjut."

oof_true = train_fe.loc[results_slot2["oof_mask"], TARGET].values
oof_pred = results_slot2["oof_preds"][results_slot2["oof_mask"]]
pos_col  = train_fe.loc[results_slot2["oof_mask"], "nama_pos"].values

# Sanity check dtype & shape sebelum hitung apa pun
assert oof_true.dtype.kind == 'f' and oof_pred.dtype.kind == 'f', "dtype tidak sesuai!"
assert oof_true.shape == oof_pred.shape == pos_col.shape, "shape mismatch!"
print(f"n_obs OOF total: {len(oof_true)}")
print(f"oof_true range : [{oof_true.min():.3f}, {oof_true.max():.3f}]")
print(f"oof_pred range : [{oof_pred.min():.3f}, {oof_pred.max():.3f}]")

full_rmse = np.sqrt(mean_squared_error(oof_true, oof_pred))
print(f"\nOOF RMSE (semua 30 pos, sanity recompute) : {full_rmse:.4f}")
assert 0.5 < full_rmse < 3.0, f"RMSE {full_rmse:.4f} di luar rentang wajar — cek ulang data!"

# ── Exclude pos outlier ──
mask_excl = ~np.isin(pos_col, OUTLIER_POS)
rmse_excl = np.sqrt(mean_squared_error(oof_true[mask_excl], oof_pred[mask_excl]))
print(f"OOF RMSE (exclude {OUTLIER_POS}) : {rmse_excl:.4f}")
print(f"Delta akibat exclude 2 pos ini   : {full_rmse - rmse_excl:+.4f}")

# ── Kontribusi tiap pos ke total SSE ──
sq_err = (oof_true - oof_pred) ** 2
contrib_df = pd.DataFrame({"pos": pos_col, "sq_err": sq_err}).groupby("pos").agg(
    n_obs=("sq_err", "count"),
    sum_sq_err=("sq_err", "sum"),
)
contrib_df["pct_of_total_sse"] = contrib_df["sum_sq_err"] / contrib_df["sum_sq_err"].sum() * 100
contrib_df["rmse"] = np.sqrt(contrib_df["sum_sq_err"] / contrib_df["n_obs"])
contrib_df = contrib_df.sort_values("pct_of_total_sse", ascending=False)

print("\n=== Kontribusi tiap pos ke total SSE (top 10) ===")
print(contrib_df.head(10).round(4).to_string())
print(f"\nTotal % SSE dari top 5 pos: {contrib_df['pct_of_total_sse'].head(5).sum():.1f}%")
print("(Kalau jauh > 5/30 = 16.7%, error terkonsentrasi di segelintir pos)")

print("\n=== Cek khusus pos outlier ===")
print(contrib_df.loc[contrib_df.index.isin(OUTLIER_POS)].round(4).to_string())

n_obs OOF total: 42930
oof_true range : [0.206, 170.100]
oof_pred range : [0.525, 144.835]

OOF RMSE (semua 30 pos, sanity recompute) : 67.2940


AssertionError: RMSE 67.2940 di luar rentang wajar — cek ulang data!

## 7. Ringkasan Perbandingan Slot 1 vs Slot 2

In [ ]:
summary = pd.DataFrame([
    {"slot": "Slot 1 (dengan nino_34)", "oof_rmse": 1.3052, "lb_score": 1.68063},
    {"slot": "Slot 2 (tanpa nino_34)",  "oof_rmse": results_slot2["oof_rmse"], "lb_score": None},
])
print(summary.to_string(index=False))
print("\nlb_score Slot 2 akan diisi setelah submit.")

## 8. Retrain Full Data (Slot 2) + Predict Test

Pakai seluruh `train_fe` (bukan per-fold), `FEATURES` Slot 2 (tanpa nino_34).
`num_boost_round` diambil dari rata-rata `best_iteration` tiap fold CV, dengan buffer +10% seperti exp001.

In [ ]:
mean_best_iter = int(np.mean(results_slot2["best_iterations"]) * 1.1)
print(f"Retraining full data, num_boost_round={mean_best_iter}...")

# Pos-stats dari FULL train (aman untuk inference — bukan CV)
pos_stats_full = train_fe.groupby("nama_pos")[TARGET].agg(
    tma_mean_pos="mean", tma_std_pos="std"
)
ac_full = train_fe.groupby("nama_pos").apply(
    lambda g: g.sort_values("datetime")[TARGET].dropna().autocorr(lag=1)
).rename("ac_lag1_pos")
pos_stats_full = pos_stats_full.join(ac_full)

X_full = train_fe[FEATURES].copy()
y_full = train_fe[TARGET]
for col in ["tma_mean_pos", "tma_std_pos", "ac_lag1_pos"]:
    X_full[col] = train_fe["nama_pos"].map(pos_stats_full[col]).values
for col in CAT_FEATURES:
    X_full[col] = X_full[col].astype("category")

ds_full = lgb.Dataset(X_full, label=y_full, categorical_feature=CAT_FEATURES)

final_model_slot2 = lgb.train(
    LGB_PARAMS,
    ds_full,
    num_boost_round=mean_best_iter,
)

final_model_slot2.save_model(f"model_s9_exp002_slot2_cv{results_slot2['oof_rmse']:.4f}.txt")
np.save("oof_exp002_slot2.npy", results_slot2["oof_preds"])
print("Model saved.")

In [ ]:
# ── Predict test ──
X_test = test_fe[FEATURES].copy()
for col in ["tma_mean_pos", "tma_std_pos", "ac_lag1_pos"]:
    X_test[col] = test_fe["nama_pos"].map(pos_stats_full[col]).values
for col in CAT_FEATURES:
    X_test[col] = X_test[col].astype(pd.CategoricalDtype(categories=X_full[col].cat.categories))

test_preds_slot2 = final_model_slot2.predict(X_test)
print(f"Predicted {len(test_preds_slot2)} rows.")

## 9. Sanity Checks Sebelum Submit

In [ ]:
assert len(test_preds_slot2) == len(test_fe), "Jumlah prediksi tidak match test set!"
assert not np.isnan(test_preds_slot2).any(), "Ada NaN di prediksi!"
assert not np.isinf(test_preds_slot2).any(), "Ada Inf di prediksi!"

print("=== SUBMISSION SANITY CHECK (Slot 2) ===")
print(f"Rows       : {len(test_preds_slot2)} (expected 21780)")
print(f"Min pred   : {test_preds_slot2.min():.4f}")
print(f"Max pred   : {test_preds_slot2.max():.4f}")
print(f"Mean pred  : {test_preds_slot2.mean():.4f}")
print(f"Train target range : [{train_fe[TARGET].min():.4f}, {train_fe[TARGET].max():.4f}]")
print(f"Train target mean  : {train_fe[TARGET].mean():.4f}")

# Bandingkan dengan prediksi exp001 (Slot 1) kalau file-nya ada, untuk cek konsistensi
try:
    sub_slot1 = pd.read_csv("sub_exp001_slot1.csv")
    print(f"\nSlot 1 pred mean: {sub_slot1['tma_mdpl'].mean():.4f} (pembanding)")
except FileNotFoundError:
    print("\n(sub_exp001_slot1.csv tidak ditemukan — skip perbandingan)")

## 10. Build & Save Submission

In [ ]:
sub = pd.read_csv("sample_submission.csv")
sub["tma_mdpl"] = test_preds_slot2

sub.to_csv("sub_exp002_slot2.csv", index=False)
print("Saved: sub_exp002_slot2.csv")
print(f"Model: model_s9_exp002_slot2_cv{results_slot2['oof_rmse']:.4f}.txt")
print("\n>>> Cek submission budget (3/hari) sebelum upload ke Kaggle. <<<")

## 11. [EKSPERIMEN LOKAL — TIDAK UNTUK SUBMIT] Validasi Horizon-Bucket Split

Sebelum implementasi 3-model per horizon-bucket, validasi dulu apakah splitting ini benar-benar
menurunkan OOF dibanding model global. **Catatan penting:** kolom `horizon_days` di train selalu 0
(lihat FE pipeline cell 8 di notebook exp001), jadi split berbasis `horizon_bucket` tidak langsung
bisa dievaluasi di CV val fold dengan cara biasa — val fold justru punya variasi horizon_days=0 sampai
~240 (dihitung dari val_start), BUKAN dari kolom `horizon_days` asli train (yang selalu 0).

Gunakan proxy: `(val_dates - val_start).dt.days` sebagai horizon simulasi, lalu bagi val set jadi
3 model terpisah berdasarkan proxy ini, bandingkan OOF gabungan vs model global (Section 5).

In [ ]:
# TODO: isi setelah keputusan section 5-10 selesai dan budget submission tersedia
# Kerangka:
# 1. Untuk tiap fold, split train row ke 3 subset berdasarkan proxy horizon (perlu definisi
#    horizon simulasi di TRAINING, bukan cuma val -- misal: gunakan days_since_last_valid_tma
#    yg sudah ada, atau buat horizon sintetis dgn hide observasi TMA sebelumnya secara artifisial)
# 2. Train 3 model terpisah (near/mid/far) per fold
# 3. Gabungkan prediksi val sesuai bucket masing2 baris
# 4. Bandingkan OOF gabungan vs results_slot2['oof_rmse']
# 5. Keputusan: pakai bucket split HANYA jika OOF turun signifikan (>0.01-0.02)
pass